In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM

# 2. Load dataset (LOCAL)
file_path = r"Google_Stock_Price.csv"
data = pd.read_csv(file_path)

# Use only 'Open' price
dataset = data['Open'].values.reshape(-1,1)

# 3. Normalize data
scaler = MinMaxScaler(feature_range=(0,1))
scaled_data = scaler.fit_transform(dataset)

# 4. Create sequences (60 days → 1 prediction)
X = []
y = []

for i in range(60, len(scaled_data)):
    X.append(scaled_data[i-60:i, 0])
    y.append(scaled_data[i, 0])

X, y = np.array(X), np.array(y)

# Reshape for LSTM [samples, timesteps, features]
X = X.reshape(X.shape[0], X.shape[1], 1)

# 5. Train-test split
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# 6. Build RNN (LSTM)
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(X.shape[1],1)),
    LSTM(50),
    Dense(1)
])

# 7. Compile
model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

# 8. Train
model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

# 9. Predict
predictions = model.predict(X_test)

# Inverse scaling
predictions = scaler.inverse_transform(predictions)
y_test_actual = scaler.inverse_transform(y_test.reshape(-1,1))

# 10. Print sample predictions
print("\nSample Predictions:\n")
for i in range(5):
    print(f"Actual: {y_test_actual[i][0]:.2f}, Predicted: {predictions[i][0]:.2f}")

# 11. RMSE (optional)
rmse = np.sqrt(np.mean((predictions - y_test_actual)**2))
print("\nRMSE:", rmse)

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


81/81 ━━━━━━━━━━━━━━━━━━━━ 8s 56ms/step - loss: 0.0020
Epoch 2/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - loss: 1.2340e-04
Epoch 3/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 6s 58ms/step - loss: 1.0451e-04
Epoch 4/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 4s 47ms/step - loss: 1.0542e-04
Epoch 5/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 4s 46ms/step - loss: 1.0559e-04
Epoch 6/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 4s 52ms/step - loss: 1.0017e-04
Epoch 7/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 4s 53ms/step - loss: 8.8473e-05
Epoch 8/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - loss: 9.3004e-05
Epoch 9/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 4s 52ms/step - loss: 8.4575e-05
Epoch 10/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - loss: 8.1501e-05
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step

Sample Predictions:

Actual: 69.47, Predicted: 73.27
Actual: 72.45, Predicted: 73.15
Actual: 72.65, Predicted: 73.12
Actual: 72.49, Predicted: 73.15
Actual: 72.00, Predicted: 73.22

RMSE: 4.079217787204881
